In [ ]:
# 导入 + 加载数据
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文显示
plt.rcParams['axes.unicode_minus'] = False

# 手动录入 4 轮数据（每轮 ragas_summary.csv ）
rounds = ['baseline\n(MiniLM+800)', 'v2\n(300过调)', 'v3\n(BGE+按条)', 'v4\n(20条+3b)', 'v5\n(20条+7b)']
data = {
    'faithfulness':     [0.23, 0.20, 0.575, 0.439, 0.695],
    'answer_relevancy': [0.55, 0.27, 0.485, 0.227, 0.474],
    'context_precision':[0.39, 0.22, 0.458, 0.548, 0.479],
    'context_recall':   [0.65, 0.53, 0.778, 0.625, 0.275],
}
df = pd.DataFrame(data, index=rounds)
print(df)

In [ ]:
# 画图
df.plot(marker='o', figsize=(10, 6), title='RAGAS 指标迭代')
plt.xticks(rotation=0)
plt.ylabel('分数')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ragas_iteration.png', dpi=100)
plt.show()

In [ ]:
# 调优动作表
actions = pd.DataFrame([
    {'轮次': 'v1', '动作': 'baseline: all-MiniLM + chunk 800', '结果': 'context_precision 0.39'},
    {'轮次': 'v2', '动作': 'chunk 800→300', '结果': '过调，标题污染，precision 0.22'},
    {'轮次': 'v3', '动作': '换 BGE-small-zh + chunk 500 + 过滤短chunk', '结果': 'faithfulness 0.23→0.58'},
    {'轮次': 'v4', '动作': '按条切分 + 双路检索 + report prompt 硬约束', '结果': '20条数据 precision 0.55'},
    {'轮次': 'v5', '动作': '3B judge → 7B judge', '结果': 'faithfulness 0.44→0.69'},
])
print(actions.to_string(index=False))

## 结论

1. **chunk 粒度是 context_precision 的决定因素**：800→300→按条切分，最终从 0.39 提升至 0.55
2. **embedding 模型必须匹配语言**：all-MiniLM-L6-v2 是英文模型，换 BGE-small-zh-v1.5 后中文语义匹配能力大幅提升
3. **prompt 硬约束降低幻觉**：report_prompt 增加"禁止编造条款编号"约束后，faithfulness 显著改善
4. **judge 模型影响评估绝对值**：3B judge 判定粗糙，7B judge 更严格；建议生产环境用 GPT-4o 类强模型评判
5. **评估本身的局限性**：本地小模型当 judge 时，context_recall 会被 judge 的断言提取粒度影响